# Three Models on a Marketing File
### BAI709 Data Modelling and Machine Learning

---

**The business question.** A bank phones customers to sell a term deposit. Each call costs money, and only about one customer in nine says yes. Which customers should the bank ring first, and which model would you back to decide that?

You will compare a logistic regression, a decision tree and a random forest on the bank's contact file, using the same split and the same testing as the other notebooks, and finish with a manager note of 150 words or fewer.

---

### How this notebook works

| Step | What it is |
|---|---|
| 1 | **Load and look** |
| 2 | **Run the models** |
| 3 | **Read the output in plain English** |
| 4 | **Check against JMP** |
| 5 | **Say it to a manager**: a note of 150 words or fewer |

- Run cells in order with **Shift+Enter**. The data is in `../data/`.
- **Copilot is on.** Read what it writes and check it against a number you trust.
- **Your turn** cells are for you to change and re-run. **Stretch** is blank for quick finishers.

## Setup

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, roc_curve, accuracy_score

import warnings
warnings.filterwarnings("ignore")
pd.set_option("display.width", 140)
print("Libraries loaded.")

### The fixed settings (one cell, leave it alone)

- **Split:** 70% training, 30% test, `random_state=1`, stratified, done **before** any preparation.
- **Logistic regression:** standardised inputs, `max_iter=5000`. **Depth-5 tree:** `max_depth=5, min_samples_leaf=50`. **Forest:** 300 trees, `min_samples_leaf=5`.
- **Cross-validation:** five folds on the training rows only, scored by AUC.

In [ ]:
SPLIT = dict(test_size=0.3, random_state=1)
CV = StratifiedKFold(5, shuffle=True, random_state=1)

def make_models():
    return {
        "Logistic regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)),
        "Tree, depth 5": DecisionTreeClassifier(max_depth=5, min_samples_leaf=50, random_state=1),
        "Random forest, 300 trees": RandomForestClassifier(n_estimators=300, min_samples_leaf=5, random_state=1, n_jobs=-1),
    }

def compare(models, X_train, X_test, y_train, y_test):
    rows, test_probs = [], {}
    for name, model in models.items():
        cv = cross_val_score(model, X_train, y_train, cv=CV, scoring="roc_auc")     # training rows only
        model.fit(X_train, y_train)
        p_tr, p_te = model.predict_proba(X_train)[:, 1], model.predict_proba(X_test)[:, 1]
        test_probs[name] = p_te
        rows.append({"Model": name, "CV AUC": cv.mean(), "CV sd": cv.std(),
                     "Train AUC": roc_auc_score(y_train, p_tr), "Test AUC": roc_auc_score(y_test, p_te),
                     "Accuracy at 0.5": accuracy_score(y_test, p_te >= 0.5)})
    return pd.DataFrame(rows).set_index("Model"), test_probs
print("Settings and the comparison function are ready.")

---
# Exercise: who should the bank ring?

## Part 1: Load and look

| Step | What it is | What we do in this part |
|---|---|---|
| 1 | **Load and look** | Open the data sheet, count contacts and subscribers. |
| 2 | **Run the model** | Prepare the table without `duration`. |
| 3 | **Read the output in plain English** | Check the shapes and the baseline. |
| 4 | **Check against JMP** | Export the prepared table with a `Split` column. |
| 5 | **Say it to a manager** | Say who the bank is trying to find. |

### Step 1: Load and look

The workbook opens on a summary sheet, so we name the data sheet. `subscribed` is the outcome: yes or no.

In [ ]:
bank = pd.read_excel("../data/BankSample.xlsx", sheet_name="BankSample")
print(f"{len(bank):,} contacts; {(bank['subscribed'] == 'yes').mean():.1%} subscribed")
print(f"Columns: {', '.join(bank.columns)}")
bank.head()

### Step 2: Prepare the table

1. The outcome is `subscribed == "yes"`.
2. Leave `duration` out of the predictors.
3. Split **first**.
4. Indicator-code the category columns, dropping one level per column (the reference level), using the training columns for the test rows.

In [ ]:
def prepare_bank(data, drop=("subscribed", "duration")):
    y = (data["subscribed"] == "yes").astype(int)
    X = data.drop(columns=list(drop))
    X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, **SPLIT)    # split FIRST
    cats = X.select_dtypes(exclude="number").columns.tolist()
    X_train = pd.get_dummies(X_train, columns=cats, drop_first=True, dtype=int)
    X_test = pd.get_dummies(X_test, columns=cats, drop_first=True, dtype=int).reindex(columns=X_train.columns, fill_value=0)
    return X_train, X_test, y_train, y_test

X_train, X_test, y_train, y_test = prepare_bank(bank)
print(f"Training: {len(X_train):,} contacts, {y_train.sum()} subscribers")
print(f"Test:     {len(X_test):,} contacts, {y_test.sum()} subscribers")
print(f"Predictors after indicator coding: {X_train.shape[1]}")

### Step 3: Read the output in plain English

In [ ]:
print(f"We trained on {len(X_train):,} contacts and kept {len(X_test):,} sealed away.")
print(f"If the bank guessed that nobody subscribes it would be right {1 - y_test.mean():.1%} of the time, so accuracy will not tell the models apart.")

### Step 4: Check against JMP

This writes the prepared table with a `Split` column and a numeric `Validation` column (0 for train, 1 for test) for JMP, so JMP uses the same rows. Open it, set `subscribed` to nominal with Target Level 1, and use `Validation` as the validation column.

In [ ]:
os.makedirs("exports", exist_ok=True)
prepared = pd.concat([X_train.assign(subscribed=y_train, Split="Train", Validation=0), X_test.assign(subscribed=y_test, Split="Test", Validation=1)])
prepared.to_csv("exports/bank_prepared_for_jmp.csv", index=False)
print(f"Written exports/bank_prepared_for_jmp.csv with {len(prepared):,} rows. Do not commit this file.")

### A check that fails quietly

A colleague prepares the file in a hurry and reports that a logistic regression reaches a test AUC of about 0.88. It runs without an error. Does that look too good for predicting who says yes to a phone call? Find out why, and fix it.

In [ ]:
def quick_logistic_auc(drop):
    Xa, Xb, ya, yb = prepare_bank(bank, drop=drop)
    m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)).fit(Xa, ya)
    return roc_auc_score(yb, m.predict_proba(Xb)[:, 1])

print(f"Colleague's test AUC: {quick_logistic_auc(['subscribed']):.3f}")

### Step 5: Say it to a manager

*One sentence:* why was `duration` left out of the models?

*Your sentence here*

---
## Part 2: Compare the three models

| Step | What it is | What we do in this part |
|---|---|---|
| 1 | **Load and look** | Reuse the prepared rows. |
| 2 | **Run the model** | Run `compare` on the logistic regression, the tree and the forest. |
| 3 | **Read the output in plain English** | Read the table, looking at CV against test. |
| 4 | **Check against JMP** | Reproduce the logistic and forest AUCs in JMP. |
| 5 | **Say it to a manager** | Write the manager note. |

### Step 2: Run the comparison

**What to look for:** compare **CV AUC** with **Test AUC** for each model, then compare **Train AUC** with **Test AUC**.

In [ ]:
results, probs = compare(make_models(), X_train, X_test, y_train, y_test)
results.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 5.5))
for (name, p), colour in zip(probs.items(), ["#1A4C3B", "#E07B39", "#2E86AB"]):
    fpr, tpr, _ = roc_curve(y_test, p)
    ax.plot(fpr, tpr, color=colour, linewidth=2, label=f"{name} (AUC {roc_auc_score(y_test, p):.3f})")
ax.plot([0, 1], [0, 1], "k--", linewidth=1, label="Coin toss")
ax.set_xlabel("Non-subscribers wrongly flagged"); ax.set_ylabel("Subscribers caught")
ax.set_title("ROC curves on the test contacts"); ax.legend(loc="lower right")
plt.show()

### Step 3: Read the output in plain English

In [ ]:
L, T, F = "Logistic regression", "Tree, depth 5", "Random forest, 300 trees"
best = results["Test AUC"].idxmax()
print(f"Best test AUC: {best} ({results.loc[best, 'Test AUC']:.3f}). Logistic {results.loc[L, 'Test AUC']:.3f}, tree {results.loc[T, 'Test AUC']:.3f}, forest {results.loc[F, 'Test AUC']:.3f}.")
print(f"Cross-validated AUC tells a closer story: logistic {results.loc[L, 'CV AUC']:.3f} (sd {results.loc[L, 'CV sd']:.3f}), forest {results.loc[F, 'CV AUC']:.3f} (sd {results.loc[F, 'CV sd']:.3f}).")
gap = results.loc[F, "Test AUC"] - results.loc[F, "CV AUC"]
print(f"The forest's test AUC is {gap:.3f} above its cross-validated AUC: one split can flatter a model.")
print(f"The forest scores {results.loc[F, 'Train AUC']:.3f} on the training rows and {results.loc[F, 'Test AUC']:.3f} on the test rows, so it has memorised some of the training contacts.")
sd_f, cv_gap = results.loc[F, "CV sd"], results.loc[F, "CV AUC"] - results.loc[L, "CV AUC"]
print(f"The forest's spread across folds ({sd_f:.3f}) is {'larger' if sd_f > cv_gap else 'smaller'} than its cross-validated lead over the logistic model ({cv_gap:.3f}), so the two are hard to separate.")

### Step 4: Check against JMP

In JMP, fit a Nominal Logistic model, a Partition tree and a Bootstrap Forest for `subscribed`, using the `Validation` column as validation. Compare each validation AUC with the **Test AUC** column above. The tree will differ because JMP stops growing it by its own rules.

### Ask Copilot

One split and one set of folds is not much evidence. Ask Copilot to repeat the cross-validation three times with different shuffles.

> *Suggested prompt:* "Using scikit-learn's RepeatedStratifiedKFold with 5 folds and 3 repeats and random_state=1, score each model in my dictionary `make_models()` on X_train and y_train by roc_auc, and print the mean and standard deviation for each model."

**Check it:** the means should be close to the CV AUC column above.

In [ ]:
# Paste and run Copilot's repeated cross-validation here

### Step 5: Write the manager note

**150 words or fewer.** Say which model you would back, and what the evidence leaves unsure. Name one thing the bank must not feed the model because it would not know it before the call.

Type your note between the quotes below and run the cell: it counts your words.

In [ ]:
note = """
Write your note here.
"""
words = len(note.split())
print(f"{words} words" + ("  (over the limit: cut it down)" if words > 150 else ""))

**Commit your note.** Save the notebook, then in the terminal:

```bash
git add three-models-marketing-file.ipynb
git commit -m "manager note: marketing file models"
git push
```

### Talk it through

- If the forest costs more to run and explain but is no better in cross-validation, what would make you choose it anyway?
- What would it cost the bank to be wrong, and how would you find out?

---
## Stretch

Blank on purpose. Ask Copilot to add a gradient boosting model to the comparison, or to show how many of the top 20% of ranked contacts are subscribers for each model (lift), and check the answer against the confusion matrix.

In [ ]:
# Stretch: your idea here

---
*BAI709 Data Modelling and Machine Learning, Regent's University London*